<h1 style="text-align: center; font-size: 40px;">IMPORTS</h1>

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.interpolate import griddata
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.io import shapereader
import cartopy.io.shapereader as shpreader
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, average_precision_score, fbeta_score, make_scorer)

import pymc as pm
import arviz as az

<h1 style="text-align: center; font-size: 40px;">NIGERIA</h1>

# CHOLERA DATASET

# LOAD AND CLEAN -- EXTRACT Benin FROM HDX FILES

### SETTINGS

In [ ]:
TARGET_COUNTRY = 'Nigeria'
years = list(range(2012, 2018))

In [ ]:
def extract_country_weekly(year, country, folder='.'):
    folder = Path(folder)
    
    # Find file
    for ext in ['.xlsx', '.xls', '.csv']:
        path = folder / f"wca_cases{year}_regcholeraplatformwca{ext}"
        if path.exists():
            break
    else:
        raise FileNotFoundError(f"No HDX file found for {year}")
    
    # Read file
    if path.suffix in ['.xlsx', '.xls']:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    
    # Strip column names
    df.columns = [str(c).strip() for c in df.columns]
    
    # Drop metadata header row (row 0 contains hashtag codes)
    df = df[~df['Country'].astype(str).str.startswith('#')].copy()
    df = df.reset_index(drop=True)
    
    # Standardise country names
    df['Country'] = (df['Country'].astype(str).str.strip()
                     .replace({"Côte d'Ivoire": "Cote d'Ivoire",
                               "Ivory Coast":   "Cote d'Ivoire",
                               "Cote D'Ivoire": "Cote d'Ivoire"}))
    
    # Filter to target country
    df = df[df['Country'] == country].copy()
    
    if len(df) == 0:
        print(f"  WARNING: {country} not found in {year} file")
        print(f"  Countries available: {df['Country'].tolist()}")
        return pd.DataFrame(columns=['Week', 'Year', 'Cases', 'Epi_Week'])
    
    # Identify week columns — handles both W1 and Week1 naming
    week_cols = []
    for c in df.columns:
        c_clean = str(c).strip()
        # Match W1...W52 or Week1...Week52
        if c_clean.upper() in ['W53', 'WEEK53']:
            continue
        if c_clean.upper().startswith('WEEK') and c_clean[4:].isdigit():
            week_cols.append(c)
        elif c_clean.upper().startswith('W') and c_clean[1:].isdigit():
            week_cols.append(c)
    
    # Reshape to long format
    df_long = df.melt(id_vars='Country',
                      value_vars=week_cols,
                      var_name='Week',
                      value_name='Cases')
    
    # Extract numeric epi week
    df_long['Epi_Week'] = (df_long['Week']
                           .str.extract(r'(\d+)')
                           .astype(int))
    
    # Clean cases
    df_long['Cases'] = df_long['Cases'].replace('-', np.nan)
    df_long['Cases'] = pd.to_numeric(df_long['Cases'], errors='coerce')
    
    df_long = df_long[df_long['Epi_Week'].between(1, 52)].copy()
    df_long['Year'] = year
    df_long = df_long[['Week', 'Year', 'Cases', 'Epi_Week']]
    
    return df_long


# Load all years
yearly_frames = []

for year in years:
    df_yr = extract_country_weekly(year, TARGET_COUNTRY)
    if len(df_yr) > 0:
        yearly_frames.append(df_yr)
        print(f"{year}: {len(df_yr)} rows")
    else:
        print(f"{year}: 0 rows — skipped")

df_long = pd.concat(yearly_frames, ignore_index=True)
df_long = df_long.sort_values(['Year', 'Epi_Week']).reset_index(drop=True)

print(f"\nFinal shape: {df_long.shape}")
print(f"Years: {sorted(df_long['Year'].unique())}")
print(df_long.head(5))

### Checking for Completeness and Zeros

In [ ]:
print("=== MISSING VALUES PER YEAR ===")
print(df_long.groupby('Year')['Cases'].apply(
    lambda x: x.isnull().sum()))

print("\n=== ZERO WEEKS PER YEAR ===")
for year in sorted(df_long['Year'].unique()):
    group = df_long[df_long['Year'] == year]
    zeros = (group['Cases'] == 0).sum()
    total = group['Cases'].notna().sum()
    print(f"{year}: {zeros} zero weeks out of {total}")

print("\n=== TOTAL CASES PER YEAR ===")
for year in sorted(df_long['Year'].unique()):
    total = df_long[df_long['Year'] == year]['Cases'].sum()
    print(f"{year}: {total:.0f} cases")

print("\n=== ROW COUNT PER YEAR ===")
print(df_long.groupby('Year')['Epi_Week'].count())

### FLAG REPORTING GAPS

In [ ]:
# A year where all 52 weeks report zero cases is treated as a surveillance reporting gap, not a genuine zero transmission
year_totals = df_long.groupby('Year')['Cases'].sum()
gap_years = year_totals[year_totals == 0].index.tolist()
print(f"Reporting gap years identified: {gap_years}")
for gap_year in gap_years:
    mask = df_long['Year'] == gap_year
    df_long.loc[mask, 'Cases'] = np.nan
    print(f"  {gap_year}: {mask.sum()} rows set to NaN")
print(f"\nTotal NaN cases after flagging: {df_long['Cases'].isna().sum()}")

### EPI-WEEK MEDIAN THRESHOLD

In [ ]:
epi_medians = df_long.groupby('Epi_Week')['Cases'].median()

df_long['Epi_Week_Median'] = df_long['Epi_Week'].map(epi_medians)
df_long['Outbreak'] = (df_long['Cases'] > df_long['Epi_Week_Median']).astype(int)

print(df_long['Outbreak'].value_counts())
print(f"Outbreak rate: {df_long['Outbreak'].mean():.1%}")

## Checking Outbreak Distribution by Year
print(df_long.groupby('Year')['Outbreak'].sum())

# TEMPERATURE AND PRECIPITATION DATASET

### SETTINGS (NASA POWER)

In [ ]:
# Folder where all your NASA POWER files are saved
data_folder = "." 

regions = ['NW', 'NE', 'SW', 'SE']
years   = [2012, 2013, 2014, 2015, 2016, 2017]

### FUNCTIONS: This will process one region-year file

In [ ]:
def process_climate_file(filepath, variable, year):
    """
    Load one NASA POWER daily file, average across grid points, convert DOY to epi week, and aggregate to weekly values.
    Temperature is aggregated as a weekly mean.
    Precipitation is aggregated as a weekly total (sum).
    """
    df = pd.read_csv(filepath, skiprows=9)
    
    # Replacing -999 missing value placeholders with NaN
    df[variable] = df[variable].replace(-999, np.nan)
    
    # Averaging across all grid points per day
    daily = df.groupby('DOY')[variable].mean().reset_index()
    
    # Converting DOY to Epi Week (clipping leap year day to week 52)
    daily['Epi_Week'] = ((daily['DOY'] - 1) // 7 + 1).clip(upper=52)
    
    # Aggregate to weekly
    if variable == 'T2M':
        weekly = daily.groupby('Epi_Week')[variable].mean().reset_index()
        weekly.columns = ['Epi_Week', 'Mean_Temp']
    else:
        weekly = daily.groupby('Epi_Week')[variable].sum().reset_index()
        weekly.columns = ['Epi_Week', 'Total_Precip']
    
    weekly['Year'] = year
    return weekly

### Looping through all regions and all years

In [ ]:
all_temp = []
all_prec = []

for year in years:
    year_temp_list = []
    year_prec_list = []
    
    for region in regions:
        temp_file = os.path.join(data_folder, f"nigeria_{region}_temp_daily_{year}.csv")
        prec_file = os.path.join(data_folder, f"nigeria_{region}_prec_daily_{year}.csv")
        
        # Processing temperature
        if os.path.exists(temp_file):
            t = process_climate_file(temp_file, 'T2M', year)
            t['Region'] = region
            year_temp_list.append(t)
        else:
            print(f"MISSING: {temp_file}")
        
        # Processing precipitation
        if os.path.exists(prec_file):
            p = process_climate_file(prec_file, 'PRECTOTCORR', year)
            p['Region'] = region
            year_prec_list.append(p)
        else:
            print(f"MISSING: {prec_file}")
    
    # Combining all 4 regions for this year and averageing across regions
    if year_temp_list:
        year_temp = pd.concat(year_temp_list)
        year_temp = year_temp.groupby(['Year', 'Epi_Week'])['Mean_Temp'].mean().reset_index()
        all_temp.append(year_temp)
    
    if year_prec_list:
        year_prec = pd.concat(year_prec_list)
        year_prec = year_prec.groupby(['Year', 'Epi_Week'])['Total_Precip'].mean().reset_index()
        all_prec.append(year_prec)

### Combining all years

In [ ]:
df_temp = pd.concat(all_temp).reset_index(drop=True)
df_prec = pd.concat(all_prec).reset_index(drop=True)

### Combining Temperature and Precipitation

In [ ]:
climate_weekly = pd.merge(df_temp, df_prec, on=['Year', 'Epi_Week'])
print(climate_weekly.shape)

print("\n=== SAMPLE ===")
print(climate_weekly.head(10))

print("\n=== MISSING VALUES ===")
print(climate_weekly.isnull().sum())

print("\n=== YEAR COVERAGE ===")
print(climate_weekly.groupby('Year')['Epi_Week'].count())

### Merging on Year and Epi_Week 

In [ ]:
df_model = pd.merge(df_long, climate_weekly, on=['Year', 'Epi_Week'])

# Drop only columns that exist
cols_to_drop = [c for c in ['Week', 'Epi_Week_Median']
                if c in df_model.columns]
df_model = df_model.drop(columns=cols_to_drop)

print(df_model.shape)
print("\n=== PREVIEW ===")
print(df_model.head(10))
print("\n=== MISSING VALUES ===")
print(df_model.isnull().sum())

# EXPLORATORY

In [ ]:
print("=== DESCRIPTIVE STATISTICS BY YEAR ===")
summary = df_long.groupby('Year')['Cases'].agg(Weeks='count', Mean='mean', Median='median', Std='std',
                                               Min='min', Max='max', Total='sum').round(1)
print(summary.to_string())

print("\n=== ZERO WEEKS BY YEAR ===")
for year in sorted(df_long['Year'].unique()):
    group = df_long[df_long['Year'] == year]
    zeros = (group['Cases'] == 0).sum()
    total = len(group)
    print(f"  {year}: {zeros} zero weeks out of {total} "
          f"({zeros/total:.0%})")

print("\n=== OVERALL WEEKLY STATISTICS ===")
print(f"  Total observations:   {len(df_long)}")
print(f"  Total cases (sum):    {df_long['Cases'].sum():,.0f}")
print(f"  Overall mean (weekly):{df_long['Cases'].mean():.1f} cases")
print(f"  Overall median:       {df_long['Cases'].median():.1f} cases")
print(f"  Overall std:          {df_long['Cases'].std():.1f}")
print(f"  Max single week:      {df_long['Cases'].max():,} cases "
      f"(Year {df_long.loc[df_long['Cases'].idxmax(), 'Year']}, "
      f"Week {df_long.loc[df_long['Cases'].idxmax(), 'Epi_Week']})")

### Weekly Cholera Cases Over Time (2012–2017)

In [ ]:
# Creating a continuous time index for plotting
df_plot = df_long.copy()
df_plot['Time_Index'] = (df_plot['Year'] - 2012) * 52 + df_plot['Epi_Week']

year_colors = {2012: '#1f77b4', 2013: '#ff7f0e', 2014: '#d62728', 2015: '#2ca02c', 2016: '#9467bd', 2017: '#8c564b'}

fig, ax = plt.subplots(figsize=(14, 5))

for year in sorted(df_plot['Year'].unique()):
    group = df_plot[df_plot['Year'] == year]
    ax.plot(group['Time_Index'], group['Cases'], color=year_colors[year], linewidth=1.2, label=str(year))

# Adding year boundary lines
for y in range(1, 6):
    ax.axvline(x=y * 52, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

# Labeling years on x axis
year_ticks = [26 + y * 52 for y in range(6)]
year_labels = [str(y) for y in range(2012, 2018)]
ax.set_xticks(year_ticks)
ax.set_xticklabels(year_labels, fontsize=11)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Weekly Reported Cases', fontsize=12)
ax.set_title('Weekly Cholera Cases in Nigeria (2012-2017)', fontsize=13)
ax.legend(title='Year', fontsize=10, loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('nigeria_weekly_cases_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### Annual Case Panels (Within-Year Variation)

In [ ]:
# Separating panel per year to show within-year seasonal variation without 2014 dominating the scale
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, year in enumerate(sorted(df_long['Year'].unique())):
    group = df_long[df_long['Year'] == year]
    axes[i].plot(group['Epi_Week'], group['Cases'], color=year_colors[year], linewidth=1.5, marker='o', markersize=3)
    axes[i].set_title(f'{year}  (Total: {group["Cases"].sum():,} cases)', fontsize=11)
    axes[i].set_ylabel('Cases', fontsize=10)
    axes[i].grid(True, linestyle='--', alpha=0.4)

    # Marking the within-year median threshold
    axes[i].axhline(y=group['Cases'].median(), color='black', linestyle='--', linewidth=1, label='Within-year median')

for ax in axes[-2:]:
    ax.set_xlabel('Epi Week', fontsize=10)

axes[0].legend(fontsize=9)
plt.suptitle('Weekly Cholera Cases by Year — Nigeria (2012–2017)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('nigeria_weekly_cases_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

### Seasonal Pattern of Cholera Transmission

In [ ]:
# Averaging cases per epi week across all years. This will reveal the seasonal transmission cycle
seasonal = df_long.groupby('Epi_Week')['Cases'].agg(Mean='mean',Median='median').reset_index()

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(seasonal['Epi_Week'], seasonal['Mean'], color='steelblue', alpha=0.7, label='Mean cases')
ax.plot(seasonal['Epi_Week'], seasonal['Median'], color='tomato', linewidth=2, marker='o', markersize=3, label='Median cases (threshold basis)')

# Shading approximate dry season
ax.axvspan(1, 13,  alpha=0.08, color='red', label='Dry season (approx)')
ax.axvspan(40, 52, alpha=0.08, color='red')

ax.set_xlabel('Epi Week', fontsize=12)
ax.set_ylabel('Cholera Cases', fontsize=12)
ax.set_title('Average Weekly Cholera Cases by Epi Week — Nigeria (2012–2017)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4, axis='y')
plt.tight_layout()
plt.savefig('nigeria_seasonal_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

### Class Distribution (Outbreak vs Non-Outbreak Weeks)

In [ ]:
outbreak_count = int(df_long['Outbreak'].sum())
non_outbreak_count = int((df_long['Outbreak'] == 0).sum())

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(['Non-Outbreak (0)', 'Outbreak (1)'], [non_outbreak_count, outbreak_count], color=['steelblue', 'tomato'], width=0.5)

for bar, val in zip(bars, [non_outbreak_count, outbreak_count]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, str(val), ha='center', va='bottom', fontsize=12)

ax.set_ylabel('Number of Epi-Week Observations', fontsize=12)
ax.set_title('Class Distribution: Outbreak vs Non-Outbreak Weeks\n' 'Nigeria Weekly Data (2012–2017)', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('nigeria_class_distribution_weekly.png', dpi=300, bbox_inches='tight')
plt.show()

### Weekly Temperature and Precipitation Trends (2012–2017)

In [ ]:
# Creating a continuous time index for climate data
climate_plot = climate_weekly.copy()
climate_plot['Time_Index'] = (climate_plot['Year'] - 2012) * 52 + climate_plot['Epi_Week']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Temperature
axes[0].plot(climate_plot['Time_Index'], climate_plot['Mean_Temp'], color='tomato', linewidth=1.2)
axes[0].set_ylabel('Mean Temperature (°C)', fontsize=12)
axes[0].set_title('Weekly Mean Temperature — Nigeria (2012–2017)', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.4)

# Precipitation
axes[1].plot(climate_plot['Time_Index'], climate_plot['Total_Precip'], color='steelblue', linewidth=1.2)
axes[1].set_ylabel('Total Precipitation (mm)', fontsize=12)
axes[1].set_title('Weekly Total Precipitation — Nigeria (2012–2017)', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.4)

# Year boundary lines on both panels
for y in range(1, 6):
    for ax in axes:
        ax.axvline(x=y * 52, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

# Year labels
year_ticks = [26 + y * 52 for y in range(6)]
year_labels = [str(y) for y in range(2012, 2018)]
axes[1].set_xticks(year_ticks)
axes[1].set_xticklabels(year_labels, fontsize=11)
axes[1].set_xlabel('Year', fontsize=12)

plt.tight_layout()
plt.savefig('nigeria_weekly_climate_trends.png', dpi=300, bbox_inches='tight')
plt.show()

### Descriptive Statistics for Weekly Climate Variables

In [ ]:
print("=== WEEKLY TEMPERATURE (°C) ===")
temp_stats = climate_weekly.groupby('Year')['Mean_Temp'].agg(Mean='mean', Min='min', Max='max', Std='std').round(2)
print(temp_stats.to_string())

print("\n=== WEEKLY PRECIPITATION (mm) ===")
prec_stats = climate_weekly.groupby('Year')['Total_Precip'].agg(Mean='mean', Min='min', Max='max', Std='std', Total='sum').round(1)
print(prec_stats.to_string())

print("\n=== OVERALL CLIMATE SUMMARY ===")
print(f"  Temperature — Mean: {climate_weekly['Mean_Temp'].mean():.2f}°C  "
      f"Range: {climate_weekly['Mean_Temp'].min():.2f} to "
      f"{climate_weekly['Mean_Temp'].max():.2f}°C")
print(f"  Precipitation — Mean: {climate_weekly['Total_Precip'].mean():.1f}mm/week  "
      f"Range: {climate_weekly['Total_Precip'].min():.1f} to "
      f"{climate_weekly['Total_Precip'].max():.1f}mm")

## Spatial Maps: Temperature and Precipitation (Nigeria 2012–2017)

### LOAD RAW GRID DATA FROM DAILY FILES

In [ ]:
# For the spatial map, I need LAT, LON and the climate value per grid point, so I average across all years and all days to get one value per grid point
temp_parts = []
prec_parts = []

for region in regions:
    for year in years:
        temp_file = f"nigeria_{region}_temp_daily_{year}.csv"
        prec_file = f"nigeria_{region}_prec_daily_{year}.csv"

        if os.path.exists(temp_file):
            t = pd.read_csv(temp_file, skiprows=9)
            t = t[t['T2M'] != -999]
            temp_parts.append(t[['LAT', 'LON', 'T2M']])

        if os.path.exists(prec_file):
            p = pd.read_csv(prec_file, skiprows=9)
            p = p[p['PRECTOTCORR'] != -999]
            prec_parts.append(p[['LAT', 'LON', 'DOY', 'PRECTOTCORR']])

# Temperature: mean T2M per grid point across all days and years
temp_all_raw = pd.concat(temp_parts)
temp_grid = temp_all_raw.groupby(['LAT', 'LON'])['T2M'].mean().reset_index()
temp_grid.columns = ['LAT', 'LON', 'ANN']

# Precipitation: sum per year per grid point, then average across years. This will give the mean annual precipitation per grid point
prec_all_raw = pd.concat(prec_parts)
prec_annual  = prec_all_raw.groupby(['LAT', 'LON', 'DOY'])['PRECTOTCORR'].mean().reset_index()
prec_grid    = prec_annual.groupby(['LAT', 'LON'])['PRECTOTCORR'].sum().reset_index()
prec_grid.columns = ['LAT', 'LON', 'ANN']

print(f"Temperature grid points: {len(temp_grid)}")
print(f"Precipitation grid points: {len(prec_grid)}")

### MAP SETTINGS

In [ ]:
shpfilename = shpreader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
reader = shpreader.Reader(shpfilename)

lon_formatter = LongitudeFormatter(number_format='.0f', degree_symbol='\N{DEGREE SIGN}', dateline_direction_label=True)
lat_formatter = LatitudeFormatter(number_format='.0f', degree_symbol='\N{DEGREE SIGN}')

# Nigeria extent
extent = [2.5, 15.0, 4.0, 14.0]

### BACKGROUND HELPER

In [ ]:
def plot_background_nigeria(ax):
    for record in reader.records():
        if record.attributes['NAME_EN'] == 'Nigeria': ax.add_geometries([record.geometry], ccrs.PlateCarree(),
                                                                        facecolor='none', edgecolor='blue', linewidth=1.5, zorder=6)
    ax.add_feature(cf.COASTLINE.with_scale('10m'), linewidth=0.4, edgecolor='black', zorder=4)
    ax.add_feature(cf.STATES.with_scale('10m'), linewidth=0.4, edgecolor='dimgray', zorder=3)
    ax.add_feature(cf.BORDERS.with_scale('10m'), linewidth=0.4, edgecolor='gray', zorder=4)
    ax.add_feature(cf.OCEAN, alpha=0.3, color='lightblue', zorder=2)
    ax.add_feature(cf.RIVERS, alpha=0.4, zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    xticks = np.round(np.linspace(extent[0], extent[1], 4), 0).tolist()
    yticks = np.round(np.linspace(extent[2], extent[3], 4), 0).tolist()
    ax.set_xticks(xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(lon_formatter)
    ax.yaxis.set_major_formatter(lat_formatter)
    ax.set_xlabel('Longitude', fontsize=10)
    ax.set_ylabel('Latitude', fontsize=10)
    ax.xaxis.set_tick_params(labelsize=9)
    ax.yaxis.set_tick_params(labelsize=9)
    return ax

### MAP PLOTTING FUNCTION

In [ ]:
def plot_nigeria_map(grid_df, title, colormap, unit, filename,
                     vmin, vmax):
    lons = grid_df['LON'].values
    lats = grid_df['LAT'].values
    vals = grid_df['ANN'].values

    lon_grid = np.linspace(extent[0], extent[1], 300)
    lat_grid = np.linspace(extent[2], extent[3], 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

    # Interpolate grid values
    vals_linear  = griddata((lons, lats), vals, (lon_mesh, lat_mesh), method='linear')
    vals_nearest = griddata((lons, lats), vals, (lon_mesh, lat_mesh), method='nearest')
    vals_final   = np.where(np.isnan(vals_linear), vals_nearest, vals_linear)

    fig, ax = plt.subplots(figsize=(8, 7),  subplot_kw={'projection': ccrs.PlateCarree()})

    im = ax.pcolormesh(lon_mesh, lat_mesh, vals_final, cmap=colormap, vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree(),
                       shading='auto', zorder=1)

    plot_background_nigeria(ax)

    cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.04, pad=0.06)
    cbar.set_label(unit, fontsize=11)

    ax.set_title(f'Nigeria — {title}\n(Weekly Climate Data, 2012–2017)', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### PLOT TEMPERATURE MAP

In [ ]:
plot_nigeria_map(grid_df  = temp_grid,  title = 'Mean Temperature', colormap = 'YlOrRd', unit = 'Temperature (°C)',
                 filename = 'nigeria_weekly_temp_map.png', vmin = temp_grid['ANN'].min(),  vmax = temp_grid['ANN'].max())

### PLOT PRECIPITATION MAP

In [ ]:
plot_nigeria_map(grid_df=prec_grid, title='Mean Annual Total Precipitation', colormap='YlGnBu', unit='Precipitation (mm/year)', 
               filename='nigeria_weekly_prec_map.png', vmin=prec_grid['ANN'].min(), vmax=prec_grid['ANN'].max())

# Lagged Features (CONTINUOUS ACROSS YEARS)

In [ ]:
# Sort chronologically across the full study period. This makes sure week 1 of 2015 can draw on week 52 of 2014
df_model = df_model.sort_values(['Year', 'Epi_Week']).reset_index(drop=True)

# Cholera case count: 1 week lag
df_model['Cases_Lag1']  = df_model['Cases'].shift(1)

# Climate: lags 1, 2 and 3 weeks for sensitivity analysis
df_model['Temp_Lag1'] = df_model['Mean_Temp'].shift(1)
df_model['Temp_Lag2'] = df_model['Mean_Temp'].shift(2)
df_model['Temp_Lag3'] = df_model['Mean_Temp'].shift(3)
df_model['Precip_Lag1'] = df_model['Total_Precip'].shift(1)
df_model['Precip_Lag2'] = df_model['Total_Precip'].shift(2)
df_model['Precip_Lag3'] = df_model['Total_Precip'].shift(3)

# Dropping only the first 3 rows globally. These have no prior values for the longest lag
df_model = df_model.dropna().reset_index(drop=True)
print({df_model.shape})

print("\n=== COLUMNS ===")
print(df_model.columns.tolist())

print("\n=== MISSING VALUES ===")
print(df_model.isnull().sum())

# Confirming the year boundary works correctly 
print("\n=== YEAR BOUNDARY CHECK ===")
boundary = df_model[df_model['Year'].isin([2014, 2015])][['Year', 'Epi_Week', 'Cases', 'Cases_Lag1', 'Precip_Lag3']].iloc[49:55]
print(boundary)

### OUTBREAK LABEL FUNCTION (TRAINING YEARS ONLY)

In [ ]:
def compute_outbreak_labels(df, train_years):
    """
    Compute epi-week medians from training years only.
    Apply those thresholds to label all rows.
    """
    train_mask = df['Year'].isin(train_years)

    # Medians computed from training years only
    train_medians = (df[train_mask].groupby('Epi_Week')['Cases'].median())

    df = df.copy()
    df['Outbreak'] = (df['Cases'] > df['Epi_Week'].map(train_medians)).astype(int)
    return df

# Quick sanity check (verifying the function works on Round 1 training years)
test_round1 = compute_outbreak_labels(df_model, train_years=[2012, 2013, 2014])

print("=== CHECK: Round 1 labels (medians from 2012-2014 only) ===")
print(test_round1.groupby('Year')['Outbreak'].agg(['sum', 'count']).round(2))
print(f"\nOverall outbreak rate in this round: {test_round1['Outbreak'].mean():.1%}")

# Evaluation Strategy

In [ ]:
# Walk-forward splits
splits = [{'train': [2012, 2013, 2014], 'test': [2015]}, {'train': [2012, 2013, 2014, 2015], 'test': [2016]},
          {'train': [2012, 2013, 2014, 2015, 2016], 'test': [2017]},]

# Primary specification
features_primary = ['Cases_Lag1', 'Temp_Lag2', 'Precip_Lag2']

# Lag sensitivity analysis
features_lag1 = ['Cases_Lag1', 'Temp_Lag1', 'Precip_Lag1']
features_lag2 = ['Cases_Lag1', 'Temp_Lag2', 'Precip_Lag2']
features_lag3 = ['Cases_Lag1', 'Temp_Lag3', 'Precip_Lag3']

# Models with class weighting
models = {'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
          'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),}

In [ ]:
def run_walk_forward(feature_set, label):
    """
    Walk-forward validation with outbreak labels computed from training years only inside each round.
    """
    print(f"\n{'='*60}")
    print(f"FEATURE SET: {label}")
    print(f"Features: {feature_set}")
    print(f"{'='*60}")
    

    all_results = []

    for model_name, model in models.items():
        print(f"\n── {model_name} ──")
        round_scores = []

        for i, split in enumerate(splits):

            # Recompute outbreak labels using training years only
            df_round = compute_outbreak_labels(df_model, train_years=split['train'])

            train_mask = df_round['Year'].isin(split['train'])
            test_mask  = df_round['Year'].isin(split['test'])

            X_train = df_round.loc[train_mask, feature_set]
            X_test  = df_round.loc[test_mask,  feature_set]
            y_train = df_round.loc[train_mask, 'Outbreak']
            y_test  = df_round.loc[test_mask,  'Outbreak']

            # Prevalence in test year (AP baseline)
            outbreak = y_test.mean()

            # Scale on training data only
            scaler = StandardScaler()
            X_train_sc = scaler.fit_transform(X_train)
            X_test_sc  = scaler.transform(X_test)

            # Fit and predict
            model.fit(X_train_sc, y_train)
            y_pred = model.predict(X_test_sc)
            y_prob = model.predict_proba(X_test_sc)[:, 1]

            scores = { 'Model': model_name, 'Features': label, 'Round': i + 1, 'Test_Year': split['test'][0], 
                       'Train_Years': f"{split['train'][0]}" f"-{split['train'][-1]}", 'Outbreak':  round(outbreak, 3),
                       'F2': round(fbeta_score(y_test, y_pred, beta=2, zero_division=0), 3),
                       'AP': round(average_precision_score(y_test, y_prob), 3),
                       'F1': round(f1_score(y_test, y_pred, zero_division=0), 3),\
                       'Accuracy': round(accuracy_score(y_test, y_pred), 3),}

            round_scores.append(scores)
            all_results.append(scores)

        # Summary
        print(f"\n  {'Round':<8} {'Test':<6} {'Outbreak':<8} "
              f"{'F2':<8} {'AP':<8} {'F1':<8} {'Acc':<8}")
        for s in round_scores:
            marker = " <- primary" if s['Round'] == 3 else ""
            print(f"  Round {s['Round']:<3} "
                  f"{s['Test_Year']:<6} "
                  f"{s['Outbreak']:<8} "
                  f"{s['F2']:<8} "
                  f"{s['AP']:<8} "
                  f"{s['F1']:<8} "
                  f"{s['Accuracy']:<8}"
                  f"{marker}")

    return pd.DataFrame(all_results)

In [ ]:
# Run both feature sets
results_primary = run_walk_forward(features_primary, 'Primary — Lag-2 Climate')
results_lag1 = run_walk_forward(features_lag1, 'Sensitivity — Lag-1 Climate')
results_lag3 = run_walk_forward(features_lag3, 'Sensitivity — Lag-3 Climate')

### COMBINED RESULTS TABLE

In [ ]:
results_all = pd.concat([results_primary, results_lag1, results_lag3]).reset_index(drop=True)

print("\n\n=== FULL RESULTS TABLE ===")
print(results_all.to_string(index=False))

### FULL-DATA REFIT FOR INTERPRETATION (PRIMARY FEATURES ONLY)

In [ ]:
# Recompute labels using all years except 2017 as training basis
df_refit = compute_outbreak_labels(df_model, train_years=[2012, 2013, 2014, 2015, 2016])
y_refit = df_refit['Outbreak']

### Logistic Regression (Coefficient Extraction)

In [ ]:
# Fitting scaler on full dataset
scaler_lr = StandardScaler()
X_all_scaled = scaler_lr.fit_transform(df_model[features_primary])

# Fitting logistic regression on full dataset
lr_full = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_full.fit(X_all_scaled, y_refit)

# Extracting coefficients
lr_coef = pd.DataFrame({'Feature': features_primary, 'Coefficient': lr_full.coef_[0]}).sort_values('Coefficient', ascending=False)

print("=== LOGISTIC REGRESSION COEFFICIENTS (Full Data Refit) ===")
print(lr_coef.to_string(index=False))

### Random Forest (Feature Importance Extraction)

In [ ]:
# Fitting scaler on full dataset
scaler_rf = StandardScaler()
X_all_scaled_rf = scaler_rf.fit_transform(df_model[features_primary])

# Fitting random forest on full dataset
rf_full = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_full.fit(X_all_scaled_rf, y_refit)

# Extracting feature importances
rf_imp = pd.DataFrame({'Feature': features_primary, 'Importance': rf_full.feature_importances_}).sort_values('Importance', ascending=False)

print("=== RANDOM FOREST FEATURE IMPORTANCES (Full Data Refit) ===")
print(rf_imp.to_string(index=False))

# BAYESIAN LOGISTIC REGRESSION (Train 2012-2016, Test 2017)

In [ ]:
# Recomputing outbreak labels using 2012-2016 training years only
df_bayes = compute_outbreak_labels(df_model, train_years=[2012, 2013, 2014, 2015, 2016])

### TRAIN/TEST SPLIT

In [ ]:
# Training on 2012-2016, test on 2017
train_mask = df_bayes['Year'].isin([2012, 2013, 2014, 2015, 2016])
test_mask  = df_bayes['Year'] == 2017

X_train_b = df_bayes.loc[train_mask, features_primary].values
X_test_b  = df_bayes.loc[test_mask,  features_primary].values
y_train_b = df_bayes.loc[train_mask, 'Outbreak'].values
y_test_b  = df_bayes.loc[test_mask,  'Outbreak'].values

print(f"Training rows:              {X_train_b.shape[0]}")
print(f"Test rows:                  {X_test_b.shape[0]}")
print(f"Training outbreak rate:     {y_train_b.mean():.1%}")
print(f"Test outbreak rate:         {y_test_b.mean():.1%}")
print(f"AP baseline for 2017:       {y_test_b.mean():.3f}")

### STANDARDISE

In [ ]:
# Fit scaler on training set only
scaler_bayes = StandardScaler()
X_train_b_sc = scaler_bayes.fit_transform(X_train_b)
X_test_b_sc  = scaler_bayes.transform(X_test_b)

### BUILD AND SAMPLE THE MODEL

In [ ]:
with pm.Model() as bayes_model:

    # Using weakly informative priors, which is Normal(0, 10), on all coefficients
    intercept = pm.Normal('Intercept', mu=0, sigma=10)
    b_cases = pm.Normal('beta_cases', mu=0, sigma=10)
    b_temp = pm.Normal('beta_temp', mu=0, sigma=10)
    b_precip = pm.Normal('beta_precip', mu=0, sigma=10)

    # Linear predictor
    mu = (intercept + b_cases * X_train_b_sc[:, 0] + b_temp * X_train_b_sc[:, 1] + b_precip * X_train_b_sc[:, 2])

    # Likelihood (binary outbreak outcome)
    p = pm.math.sigmoid(mu)
    likelihood = pm.Bernoulli('likelihood', p=p, observed=y_train_b)

    # Sample using NUTS
    trace = pm.sample(1000, tune=1000, chains=4, target_accept=0.9, random_seed=42, progressbar=True)

### CONVERGENCE CHECK

In [ ]:
print("\n=== GELMAN-RUBIN R-HAT VALUES ===")
rhat = az.rhat(trace)
for var in ['Intercept', 'beta_cases', 'beta_temp', 'beta_precip']:
    print(f"  {var}: {float(rhat[var].values):.4f}")

### POSTERIOR SUMMARY

In [ ]:
print("\n=== POSTERIOR SUMMARY ===")
summary = az.summary(trace, var_names=['Intercept', 'beta_cases', 'beta_temp', 'beta_precip'], hdi_prob=0.95)
print(summary[['mean', 'hdi_2.5%', 'hdi_97.5%']])

### PREDICTION ON TEST SET (2017)

In [ ]:
# Extract posterior means as point estimates
post = trace.posterior
b0  = float(post['Intercept'].values.mean())
b1  = float(post['beta_cases'].values.mean())
b2  = float(post['beta_temp'].values.mean())
b3  = float(post['beta_precip'].values.mean())

# Compute predicted probabilities on 2017 test set
logits   = (b0 + b1 * X_test_b_sc[:, 0] + b2 * X_test_b_sc[:, 1] + b3 * X_test_b_sc[:, 2])
y_prob_b = 1 / (1 + np.exp(-logits))
y_pred_b = (y_prob_b >= 0.5).astype(int)

##### EVALUATE ON TEST SET

In [ ]:
print("\n=== BAYESIAN MODEL PERFORMANCE (Test: 2017) ===")
print(f"AP baseline (outbreak):   {y_test_b.mean():.3f}")
print(f"F2:                         "
      f"{fbeta_score(y_test_b, y_pred_b, beta=2, zero_division=0):.3f}")
print(f"AP:                         "
      f"{average_precision_score(y_test_b, y_prob_b):.3f}")
print(f"F1:                         "
      f"{f1_score(y_test_b, y_pred_b, zero_division=0):.3f}")
print(f"Accuracy:                   "
      f"{accuracy_score(y_test_b, y_pred_b):.3f}")

<h1 style="text-align: center; font-size: 40px;">BENIN</h1>

# CHOLERA DATASET

# LOAD AND CLEAN -- EXTRACT Benin FROM HDX FILES

### SETTINGS

In [ ]:
TARGET_COUNTRY = 'Benin'
years = list(range(2012, 2018))

In [ ]:
def extract_country_weekly(year, country, folder='.'):
    folder = Path(folder)
    
    # Find file
    for ext in ['.xlsx', '.xls', '.csv']:
        path = folder / f"wca_cases{year}_regcholeraplatformwca{ext}"
        if path.exists():
            break
    else:
        raise FileNotFoundError(f"No HDX file found for {year}")
    
    # Read file
    if path.suffix in ['.xlsx', '.xls']:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    
    # Strip column names
    df.columns = [str(c).strip() for c in df.columns]
    
    # Drop metadata header row (row 0 contains hashtag codes)
    df = df[~df['Country'].astype(str).str.startswith('#')].copy()
    df = df.reset_index(drop=True)
    
    # Standardise country names
    df['Country'] = (df['Country'].astype(str).str.strip()
                     .replace({"Côte d'Ivoire": "Cote d'Ivoire",
                               "Ivory Coast":   "Cote d'Ivoire",
                               "Cote D'Ivoire": "Cote d'Ivoire"}))
    
    # Filter to target country
    df = df[df['Country'] == country].copy()
    
    if len(df) == 0:
        print(f"  WARNING: {country} not found in {year} file")
        print(f"  Countries available: {df['Country'].tolist()}")
        return pd.DataFrame(columns=['Week', 'Year', 'Cases', 'Epi_Week'])
    
    # Identify week columns — handles both W1 and Week1 naming
    week_cols = []
    for c in df.columns:
        c_clean = str(c).strip()
        # Match W1...W52 or Week1...Week52
        if c_clean.upper() in ['W53', 'WEEK53']:
            continue
        if c_clean.upper().startswith('WEEK') and c_clean[4:].isdigit():
            week_cols.append(c)
        elif c_clean.upper().startswith('W') and c_clean[1:].isdigit():
            week_cols.append(c)
    
    # Reshape to long format
    df_long = df.melt(id_vars='Country',
                      value_vars=week_cols,
                      var_name='Week',
                      value_name='Cases')
    
    # Extract numeric epi week
    df_long['Epi_Week'] = (df_long['Week']
                           .str.extract(r'(\d+)')
                           .astype(int))
    
    # Clean cases
    df_long['Cases'] = df_long['Cases'].replace('-', np.nan)
    df_long['Cases'] = pd.to_numeric(df_long['Cases'], errors='coerce')
    
    df_long = df_long[df_long['Epi_Week'].between(1, 52)].copy()
    df_long['Year'] = year
    df_long = df_long[['Week', 'Year', 'Cases', 'Epi_Week']]
    
    return df_long


# Load all years
yearly_frames = []

for year in years:
    df_yr = extract_country_weekly(year, TARGET_COUNTRY)
    if len(df_yr) > 0:
        yearly_frames.append(df_yr)
        print(f"{year}: {len(df_yr)} rows")
    else:
        print(f"{year}: 0 rows — skipped")

df_long = pd.concat(yearly_frames, ignore_index=True)
df_long = df_long.sort_values(['Year', 'Epi_Week']).reset_index(drop=True)

print(f"\nFinal shape: {df_long.shape}")
print(f"Years: {sorted(df_long['Year'].unique())}")
print(df_long.head(5))

### Checking for Completeness and Zeros

In [ ]:
print("=== MISSING VALUES PER YEAR ===")
print(df_long.groupby('Year')['Cases'].apply(
    lambda x: x.isnull().sum()))

print("\n=== ZERO WEEKS PER YEAR ===")
for year in sorted(df_long['Year'].unique()):
    group = df_long[df_long['Year'] == year]
    zeros = (group['Cases'] == 0).sum()
    total = group['Cases'].notna().sum()
    print(f"{year}: {zeros} zero weeks out of {total}")

print("\n=== TOTAL CASES PER YEAR ===")
for year in sorted(df_long['Year'].unique()):
    total = df_long[df_long['Year'] == year]['Cases'].sum()
    print(f"{year}: {total:.0f} cases")

print("\n=== ROW COUNT PER YEAR ===")
print(df_long.groupby('Year')['Epi_Week'].count())

### FLAG REPORTING GAPS

In [ ]:
# A year where all 52 weeks report zero cases is treated as a surveillance reporting gap, not a genuine zero transmission
year_totals = df_long.groupby('Year')['Cases'].sum()
gap_years = year_totals[year_totals == 0].index.tolist()
print(f"Reporting gap years identified: {gap_years}")
for gap_year in gap_years:
    mask = df_long['Year'] == gap_year
    df_long.loc[mask, 'Cases'] = np.nan
    print(f"  {gap_year}: {mask.sum()} rows set to NaN")
print(f"\nTotal NaN cases after flagging: {df_long['Cases'].isna().sum()}")

### EPI-WEEK MEDIAN THRESHOLD

In [ ]:
epi_medians = df_long.groupby('Epi_Week')['Cases'].median()

df_long['Epi_Week_Median'] = df_long['Epi_Week'].map(epi_medians)
df_long['Outbreak'] = (df_long['Cases'] > df_long['Epi_Week_Median']).astype(int)

print(df_long['Outbreak'].value_counts())
print(f"Outbreak rate: {df_long['Outbreak'].mean():.1%}")

## Checking Outbreak Distribution by Year
print(df_long.groupby('Year')['Outbreak'].sum())

# TEMPERATURE AND PRECIPITATION DATASET

### SETTINGS (NASA POWER)

In [ ]:
years_climate = [2012, 2013, 2014, 2015, 2016, 2017]

In [ ]:
def process_benin_climate_file(filepath, variable, year):
    """
    Load one Benin NASA POWER daily file.
    Average across grid points per day.
    Aggregate to epi-week resolution.
    Temperature = weekly mean.
    Precipitation = weekly total.
    """
    df = pd.read_csv(filepath, skiprows=9)

    # Replace -999 missing value placeholders
    df[variable] = df[variable].replace(-999, np.nan)

    # Average across all grid points per day
    daily = df.groupby('DOY')[variable].mean().reset_index()

    # Convert DOY to epi week (clip leap year to week 52)
    daily['Epi_Week'] = ((daily['DOY'] - 1) // 7 + 1).clip(upper=52)

    # Aggregate to weekly
    if variable == 'T2M':
        weekly = daily.groupby('Epi_Week')[variable].mean().reset_index()
        weekly.columns = ['Epi_Week', 'Mean_Temp']
    else:
        weekly = daily.groupby('Epi_Week')[variable].sum().reset_index()
        weekly.columns = ['Epi_Week', 'Total_Precip']

    weekly['Year'] = year
    return weekly

In [ ]:
all_temp = []
all_prec = []

In [ ]:
for year in years_climate:
    temp_file = f"benin_T2M_{year}.csv"
    prec_file = f"benin_PRECTOTCORR_{year}.csv"

    if os.path.exists(temp_file):
        t = process_benin_climate_file(temp_file, 'T2M', year)
        all_temp.append(t)
    else:
        print(f"MISSING: {temp_file}")

    if os.path.exists(prec_file):
        p = process_benin_climate_file(prec_file, 'PRECTOTCORR', year)
        all_prec.append(p)
    else:
        print(f"MISSING: {prec_file}")

In [ ]:
df_temp = pd.concat(all_temp).reset_index(drop=True)
df_prec = pd.concat(all_prec).reset_index(drop=True)

In [ ]:
climate_weekly = pd.merge(df_temp, df_prec, on=['Year', 'Epi_Week'])

print(f"Climate shape: {climate_weekly.shape}")
print("\n=== SAMPLE ===")
print(climate_weekly.head(10))
print("\n=== MISSING VALUES ===")
print(climate_weekly.isnull().sum())
print("\n=== YEAR COVERAGE ===")
print(climate_weekly.groupby('Year')['Epi_Week'].count())

### Merging on Year and Epi_Week 

In [ ]:
df_model = pd.merge(df_long, climate_weekly, on=['Year', 'Epi_Week'])

# Drop only columns that exist
cols_to_drop = [c for c in ['Week', 'Epi_Week_Median']
                if c in df_model.columns]
df_model = df_model.drop(columns=cols_to_drop)

print(df_model.shape)
print("\n=== PREVIEW ===")
print(df_model.head(10))
print("\n=== MISSING VALUES ===")
print(df_model.isnull().sum())

# EXPLORATORY

In [ ]:
print("=== DESCRIPTIVE STATISTICS BY YEAR ===")
summary = df_long.groupby('Year')['Cases'].agg(Weeks='count', Mean='mean', Median='median', Std='std',
                                               Min='min', Max='max', Total='sum').round(1)
print(summary.to_string())

print("\n=== ZERO WEEKS BY YEAR ===")
for year in sorted(df_long['Year'].unique()):
    group = df_long[df_long['Year'] == year]
    zeros = (group['Cases'] == 0).sum()
    total = len(group)
    print(f"  {year}: {zeros} zero weeks out of {total} "
          f"({zeros/total:.0%})")

print("\n=== OVERALL WEEKLY STATISTICS ===")
print(f"  Total observations:   {len(df_long)}")
print(f"  Total cases (sum):    {df_long['Cases'].sum():,.0f}")
print(f"  Overall mean (weekly):{df_long['Cases'].mean():.1f} cases")
print(f"  Overall median:       {df_long['Cases'].median():.1f} cases")
print(f"  Overall std:          {df_long['Cases'].std():.1f}")
print(f"  Max single week:      {df_long['Cases'].max():,} cases "
      f"(Year {df_long.loc[df_long['Cases'].idxmax(), 'Year']}, "
      f"Week {df_long.loc[df_long['Cases'].idxmax(), 'Epi_Week']})")

### Weekly Cholera Cases Over Time (2012–2017)

In [ ]:
# Creating a continuous time index for plotting
df_plot = df_long.copy()
df_plot['Time_Index'] = (df_plot['Year'] - 2012) * 52 + df_plot['Epi_Week']

year_colors = {2012: '#1f77b4', 2013: '#ff7f0e', 2014: '#d62728', 2015: '#2ca02c', 2016: '#9467bd', 2017: '#8c564b'}

fig, ax = plt.subplots(figsize=(14, 5))

for year in sorted(df_plot['Year'].unique()):
    group = df_plot[df_plot['Year'] == year]
    ax.plot(group['Time_Index'], group['Cases'], color=year_colors[year], linewidth=1.2, label=str(year))

# Adding year boundary lines
for y in range(1, 6):
    ax.axvline(x=y * 52, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

# Labeling years on x axis
year_ticks = [26 + y * 52 for y in range(6)]
year_labels = [str(y) for y in range(2012, 2018)]
ax.set_xticks(year_ticks)
ax.set_xticklabels(year_labels, fontsize=11)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Weekly Reported Cases', fontsize=12)
ax.set_title('Weekly Cholera Cases in Benin (2012-2017)', fontsize=13)
ax.legend(title='Year', fontsize=10, loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('benin_weekly_cases_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### Annual Case Panels (Within-Year Variation)

In [ ]:
# Separating panel per year to show within-year seasonal variation without 2014 dominating the scale
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, year in enumerate(sorted(df_long['Year'].unique())):
    group = df_long[df_long['Year'] == year]
    axes[i].plot(group['Epi_Week'], group['Cases'], color=year_colors[year], linewidth=1.5, marker='o', markersize=3)
    axes[i].set_title(f'{year}  (Total: {group["Cases"].sum():,} cases)', fontsize=11)
    axes[i].set_ylabel('Cases', fontsize=10)
    axes[i].grid(True, linestyle='--', alpha=0.4)

    # Marking the within-year median threshold
    axes[i].axhline(y=group['Cases'].median(), color='black', linestyle='--', linewidth=1, label='Within-year median')

for ax in axes[-2:]:
    ax.set_xlabel('Epi Week', fontsize=10)

axes[0].legend(fontsize=9)
plt.suptitle('Weekly Cholera Cases by Year — Benin (2012–2017)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('benin_weekly_cases_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

### Seasonal Pattern of Cholera Transmission

In [ ]:
# Averaging cases per epi week across all years. This will reveal the seasonal transmission cycle
seasonal = df_long.groupby('Epi_Week')['Cases'].agg(Mean='mean',Median='median').reset_index()

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(seasonal['Epi_Week'], seasonal['Mean'], color='steelblue', alpha=0.7, label='Mean cases')
ax.plot(seasonal['Epi_Week'], seasonal['Median'], color='tomato', linewidth=2, marker='o', markersize=3, label='Median cases (threshold basis)')

# Shading approximate dry season
ax.axvspan(1, 13,  alpha=0.08, color='red', label='Dry season (approx)')
ax.axvspan(40, 52, alpha=0.08, color='red')

ax.set_xlabel('Epi Week', fontsize=12)
ax.set_ylabel('Cholera Cases', fontsize=12)
ax.set_title('Average Weekly Cholera Cases by Epi Week — Benin (2012–2017)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4, axis='y')
plt.tight_layout()
plt.savefig('benin_seasonal_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

### Class Distribution (Outbreak vs Non-Outbreak Weeks)

In [ ]:
outbreak_count = int(df_long['Outbreak'].sum())
non_outbreak_count = int((df_long['Outbreak'] == 0).sum())

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(['Non-Outbreak (0)', 'Outbreak (1)'], [non_outbreak_count, outbreak_count], color=['steelblue', 'tomato'], width=0.5)

for bar, val in zip(bars, [non_outbreak_count, outbreak_count]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, str(val), ha='center', va='bottom', fontsize=12)

ax.set_ylabel('Number of Epi-Week Observations', fontsize=12)
ax.set_title('Class Distribution: Outbreak vs Non-Outbreak Weeks\n' 'Benin Weekly Data (2012–2017)', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('benin_class_distribution_weekly.png', dpi=300, bbox_inches='tight')
plt.show()

### Weekly Temperature and Precipitation Trends (2012–2017)

In [ ]:
# Creating a continuous time index for climate data
climate_plot = climate_weekly.copy()
climate_plot['Time_Index'] = (climate_plot['Year'] - 2012) * 52 + climate_plot['Epi_Week']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Temperature
axes[0].plot(climate_plot['Time_Index'], climate_plot['Mean_Temp'], color='tomato', linewidth=1.2)
axes[0].set_ylabel('Mean Temperature (°C)', fontsize=12)
axes[0].set_title('Weekly Mean Temperature — Benin (2012–2017)', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.4)

# Precipitation
axes[1].plot(climate_plot['Time_Index'], climate_plot['Total_Precip'], color='steelblue', linewidth=1.2)
axes[1].set_ylabel('Total Precipitation (mm)', fontsize=12)
axes[1].set_title('Weekly Total Precipitation — Benin (2012–2017)', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.4)

# Year boundary lines on both panels
for y in range(1, 6):
    for ax in axes:
        ax.axvline(x=y * 52, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

# Year labels
year_ticks = [26 + y * 52 for y in range(6)]
year_labels = [str(y) for y in range(2012, 2018)]
axes[1].set_xticks(year_ticks)
axes[1].set_xticklabels(year_labels, fontsize=11)
axes[1].set_xlabel('Year', fontsize=12)

plt.tight_layout()
plt.savefig('benin_weekly_climate_trends.png', dpi=300, bbox_inches='tight')
plt.show()

### Descriptive Statistics for Weekly Climate Variables

In [ ]:
print("=== WEEKLY TEMPERATURE (°C) ===")
temp_stats = climate_weekly.groupby('Year')['Mean_Temp'].agg(Mean='mean', Min='min', Max='max', Std='std').round(2)
print(temp_stats.to_string())

print("\n=== WEEKLY PRECIPITATION (mm) ===")
prec_stats = climate_weekly.groupby('Year')['Total_Precip'].agg(Mean='mean', Min='min', Max='max', Std='std', Total='sum').round(1)
print(prec_stats.to_string())

print("\n=== OVERALL CLIMATE SUMMARY ===")
print(f"  Temperature — Mean: {climate_weekly['Mean_Temp'].mean():.2f}°C  "
      f"Range: {climate_weekly['Mean_Temp'].min():.2f} to "
      f"{climate_weekly['Mean_Temp'].max():.2f}°C")
print(f"  Precipitation — Mean: {climate_weekly['Total_Precip'].mean():.1f}mm/week  "
      f"Range: {climate_weekly['Total_Precip'].min():.1f} to "
      f"{climate_weekly['Total_Precip'].max():.1f}mm")

## Spatial Maps: Temperature and Precipitation (Benin 2012–2017)

### LOAD RAW GRID DATA FROM DAILY FILES

In [ ]:
# For the spatial map, I need LAT, LON and the climate value per grid point, so I average across all years and all days to get one value per grid point
temp_parts = []
prec_parts = []

for year in years_climate:
    temp_file = f"benin_T2M_{year}.csv"
    prec_file = f"benin_PRECTOTCORR_{year}.csv"

    if os.path.exists(temp_file):
        t = pd.read_csv(temp_file, skiprows=9)
        t = t[t['T2M'] != -999]
        temp_parts.append(t[['LAT', 'LON', 'T2M']])

    if os.path.exists(prec_file):
        p = pd.read_csv(prec_file, skiprows=9)
        p = p[p['PRECTOTCORR'] != -999]
        prec_parts.append(p[['LAT', 'LON', 'DOY', 'PRECTOTCORR']])

# Temperature: mean T2M per grid point across all days and years
temp_all_raw = pd.concat(temp_parts)
temp_grid = temp_all_raw.groupby(['LAT', 'LON'])['T2M'].mean().reset_index()
temp_grid.columns = ['LAT', 'LON', 'ANN']

# Precipitation: sum per DOY then average across years
prec_all_raw = pd.concat(prec_parts)
prec_annual = prec_all_raw.groupby(
    ['LAT', 'LON', 'DOY'])['PRECTOTCORR'].mean().reset_index()
prec_grid = prec_annual.groupby(
    ['LAT', 'LON'])['PRECTOTCORR'].sum().reset_index()
prec_grid.columns = ['LAT', 'LON', 'ANN']

print(f"Temperature grid points: {len(temp_grid)}")
print(f"Precipitation grid points: {len(prec_grid)}")

### MAP SETTINGS

In [ ]:
shpfilename = shpreader.natural_earth(
    resolution='10m', category='cultural', name='admin_0_countries')
reader = shpreader.Reader(shpfilename)

lon_formatter = LongitudeFormatter(
    number_format='.0f',
    degree_symbol='\N{DEGREE SIGN}',
    dateline_direction_label=True)
lat_formatter = LatitudeFormatter(
    number_format='.0f',
    degree_symbol='\N{DEGREE SIGN}')

# Benin bounding box
extent = [0.68, 3.96, 6.20, 12.47]

### BACKGROUND HELPER

In [ ]:
def plot_background_benin(ax):
    for record in reader.records():
        if record.attributes['NAME_EN'] == 'Benin':
            ax.add_geometries(
                [record.geometry], ccrs.PlateCarree(),
                facecolor='none', edgecolor='blue',
                linewidth=1.5, zorder=6)
    ax.add_feature(cf.COASTLINE.with_scale('10m'),
                   linewidth=0.4, edgecolor='black', zorder=4)
    ax.add_feature(cf.STATES.with_scale('10m'),
                   linewidth=0.4, edgecolor='dimgray', zorder=3)
    ax.add_feature(cf.BORDERS.with_scale('10m'),
                   linewidth=0.4, edgecolor='gray', zorder=4)
    ax.add_feature(cf.OCEAN, alpha=0.3, color='lightblue', zorder=2)
    ax.add_feature(cf.RIVERS, alpha=0.4, zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    xticks = np.round(np.linspace(extent[0], extent[1], 4), 0).tolist()
    yticks = np.round(np.linspace(extent[2], extent[3], 4), 0).tolist()
    ax.set_xticks(xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(lon_formatter)
    ax.yaxis.set_major_formatter(lat_formatter)
    ax.set_xlabel('Longitude', fontsize=10)
    ax.set_ylabel('Latitude', fontsize=10)
    ax.xaxis.set_tick_params(labelsize=9)
    ax.yaxis.set_tick_params(labelsize=9)
    return ax

### MAP PLOTTING FUNCTION

In [ ]:
def plot_benin_map(grid_df, title, colormap, unit, filename, vmin, vmax):
    lons = grid_df['LON'].values
    lats = grid_df['LAT'].values
    vals = grid_df['ANN'].values
    lon_grid = np.linspace(extent[0], extent[1], 300)
    lat_grid = np.linspace(extent[2], extent[3], 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    vals_linear  = griddata((lons, lats), vals, (lon_mesh, lat_mesh), method='linear')
    vals_nearest = griddata((lons, lats), vals, (lon_mesh, lat_mesh), method='nearest')
    vals_final   = np.where(np.isnan(vals_linear), vals_nearest, vals_linear)
    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    im = ax.pcolormesh(lon_mesh, lat_mesh, vals_final, cmap=colormap, vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree(), shading='auto', zorder=1)
    plot_background_benin(ax)
    cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.04, pad=0.06)
    cbar.set_label(unit, fontsize=11)
    ax.set_title(f'Benin — {title}\n(Weekly Climate Data, 2012–2017)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### PLOT TEMPERATURE MAP

In [ ]:
plot_benin_map(grid_df=temp_grid, title='Mean Temperature', colormap='YlOrRd', unit='Temperature (°C)', 
               filename='benin_weekly_temp_map.png', vmin=temp_grid['ANN'].min(), vmax=temp_grid['ANN'].max())

### PLOT PRECIPITATION MAP

In [ ]:
plot_benin_map(grid_df=prec_grid, title='Mean Annual Total Precipitation', colormap='YlGnBu', unit='Precipitation (mm/year)', 
               filename='benin_weekly_prec_map.png', vmin=prec_grid['ANN'].min(), vmax=prec_grid['ANN'].max())

# Lagged Features (CONTINUOUS ACROSS YEARS)

In [ ]:
# Sort chronologically across the full study period. This makes sure week 1 of 2015 can draw on week 52 of 2014
df_model = df_model.sort_values(['Year', 'Epi_Week']).reset_index(drop=True)

# Cholera case count: 1 week lag
df_model['Cases_Lag1']  = df_model['Cases'].shift(1)

# Climate: lags 1, 2 and 3 weeks for sensitivity analysis
df_model['Temp_Lag1'] = df_model['Mean_Temp'].shift(1)
df_model['Temp_Lag2'] = df_model['Mean_Temp'].shift(2)
df_model['Temp_Lag3'] = df_model['Mean_Temp'].shift(3)
df_model['Precip_Lag1'] = df_model['Total_Precip'].shift(1)
df_model['Precip_Lag2'] = df_model['Total_Precip'].shift(2)
df_model['Precip_Lag3'] = df_model['Total_Precip'].shift(3)

# Dropping only the first 3 rows globally. These have no prior values for the longest lag
df_model = df_model.dropna().reset_index(drop=True)
print({df_model.shape})

print("\n=== COLUMNS ===")
print(df_model.columns.tolist())

print("\n=== MISSING VALUES ===")
print(df_model.isnull().sum())

# Confirming the year boundary works correctly 
print("\n=== YEAR BOUNDARY CHECK ===")
boundary = df_model[df_model['Year'].isin([2014, 2015])][['Year', 'Epi_Week', 'Cases', 'Cases_Lag1', 'Precip_Lag3']].iloc[49:55]
print(boundary)

### OUTBREAK LABEL FUNCTION (TRAINING YEARS ONLY)

In [ ]:
def compute_outbreak_labels(df, train_years):
    """
    Compute epi-week medians from training years only.
    Apply those thresholds to label all rows.
    """
    train_mask = df['Year'].isin(train_years)

    # Medians computed from training years only
    train_medians = (df[train_mask].groupby('Epi_Week')['Cases'].median())

    df = df.copy()
    df['Outbreak'] = (df['Cases'] > df['Epi_Week'].map(train_medians)).astype(int)
    return df

# Quick sanity check (verifying the function works on Round 1 training years)
test_round1 = compute_outbreak_labels(df_model, train_years=[2012, 2013, 2014])

print("=== CHECK: Round 1 labels (medians from 2012-2014 only) ===")
print(test_round1.groupby('Year')['Outbreak'].agg(['sum', 'count']).round(2))
print(f"\nOverall outbreak rate in this round: {test_round1['Outbreak'].mean():.1%}")

# Evaluation Strategy

In [ ]:
# Walk-forward splits
splits = [{'train': [2012, 2013, 2014], 'test': [2015]}, {'train': [2012, 2013, 2014, 2015], 'test': [2016]},
          {'train': [2012, 2013, 2014, 2015, 2016], 'test': [2017]},]

# Primary specification
features_primary = ['Cases_Lag1', 'Temp_Lag2', 'Precip_Lag2']

# Lag sensitivity analysis
features_lag1 = ['Cases_Lag1', 'Temp_Lag1', 'Precip_Lag1']
features_lag2 = ['Cases_Lag1', 'Temp_Lag2', 'Precip_Lag2']
features_lag3 = ['Cases_Lag1', 'Temp_Lag3', 'Precip_Lag3']

# Models with class weighting
models = {'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
          'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),}

In [ ]:
def run_walk_forward(feature_set, label):
    print(f"\n{'='*60}")
    print(f"FEATURE SET: {label}")
    print(f"Features: {feature_set}")
    print(f"{'='*60}")
    all_results = []
    for model_name, model in models.items():
        print(f"\n── {model_name} ──")
        round_scores = []
        for i, split in enumerate(splits):
            df_round = compute_outbreak_labels(df_model, train_years=split['train'])
            train_mask = df_round['Year'].isin(split['train'])
            test_mask  = df_round['Year'].isin(split['test'])
            X_train = df_round.loc[train_mask, feature_set]
            X_test  = df_round.loc[test_mask,  feature_set]
            y_train = df_round.loc[train_mask, 'Outbreak']
            y_test  = df_round.loc[test_mask,  'Outbreak']
            # Skip round if test set is empty
            if len(X_test) == 0:
                print(f"  Round {i+1} (Test: {split['test'][0]}): SKIPPED — no test data for this year")
                continue
            outbreak = y_test.mean()
            scaler = StandardScaler()
            X_train_sc = scaler.fit_transform(X_train)
            X_test_sc  = scaler.transform(X_test)
            model.fit(X_train_sc, y_train)
            y_pred = model.predict(X_test_sc)
            y_prob = model.predict_proba(X_test_sc)[:, 1]
            scores = {'Model': model_name, 'Features': label, 'Round': i + 1, 'Test_Year': split['test'][0], 'Train_Years': f"{split['train'][0]}-{split['train'][-1]}", 'Outbreak': round(outbreak, 3), 'F2': round(fbeta_score(y_test, y_pred, beta=2, zero_division=0), 3), 'AP': round(average_precision_score(y_test, y_prob), 3), 'F1': round(f1_score(y_test, y_pred, zero_division=0), 3), 'Accuracy': round(accuracy_score(y_test, y_pred), 3)}
            round_scores.append(scores)
            all_results.append(scores)
        print(f"\n  {'Round':<8} {'Test':<6} {'Outbreak':<10} {'F2':<8} {'AP':<8} {'F1':<8} {'Acc':<8}")
        for s in round_scores:
            marker = " <- primary" if s['Round'] == 3 else ""
            print(f"  Round {s['Round']:<3} {s['Test_Year']:<6} {s['Outbreak']:<10} {s['F2']:<8} {s['AP']:<8} {s['F1']:<8} {s['Accuracy']:<8}{marker}")
    return pd.DataFrame(all_results)

In [ ]:
# Run both feature sets
results_primary = run_walk_forward(features_primary, 'Primary — Lag-2 Climate')
results_lag1 = run_walk_forward(features_lag1, 'Sensitivity — Lag-1 Climate')
results_lag3 = run_walk_forward(features_lag3, 'Sensitivity — Lag-3 Climate')

### COMBINED RESULTS TABLE

In [ ]:
results_all = pd.concat([results_primary, results_lag1, results_lag3]).reset_index(drop=True)

print("\n\n=== FULL RESULTS TABLE ===")
print(results_all.to_string(index=False))

### FULL-DATA REFIT FOR INTERPRETATION (PRIMARY FEATURES ONLY)

In [ ]:
# Recompute labels using all years except 2017 as training basis
df_refit = compute_outbreak_labels(df_model, train_years=[2012, 2013, 2014, 2015, 2016])
y_refit = df_refit['Outbreak']

### Logistic Regression (Coefficient Extraction)

In [ ]:
# Fitting scaler on full dataset
scaler_lr = StandardScaler()
X_all_scaled = scaler_lr.fit_transform(df_model[features_primary])

# Fitting logistic regression on full dataset
lr_full = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_full.fit(X_all_scaled, y_refit)

# Extracting coefficients
lr_coef = pd.DataFrame({'Feature': features_primary, 'Coefficient': lr_full.coef_[0]}).sort_values('Coefficient', ascending=False)

print("=== LOGISTIC REGRESSION COEFFICIENTS (Full Data Refit) ===")
print(lr_coef.to_string(index=False))

### Random Forest (Feature Importance Extraction)

In [ ]:
# Fitting scaler on full dataset
scaler_rf = StandardScaler()
X_all_scaled_rf = scaler_rf.fit_transform(df_model[features_primary])

# Fitting random forest on full dataset
rf_full = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_full.fit(X_all_scaled_rf, y_refit)

# Extracting feature importances
rf_imp = pd.DataFrame({'Feature': features_primary, 'Importance': rf_full.feature_importances_}).sort_values('Importance', ascending=False)

print("=== RANDOM FOREST FEATURE IMPORTANCES (Full Data Refit) ===")
print(rf_imp.to_string(index=False))

# BAYESIAN LOGISTIC REGRESSION I (Train 2012-2016, Test 2017)

In [ ]:
# Recomputing outbreak labels using 2012-2017 training years only
df_bayes = compute_outbreak_labels(df_model, train_years=[2012, 2013, 2014, 2015,2016])

### TRAIN/TEST SPLIT

In [ ]:
# Training on 2012-2015, test on 2016
train_mask = df_bayes['Year'].isin([2012, 2013, 2014, 2015,2016])
test_mask  = df_bayes['Year'] == 2017

X_train_b = df_bayes.loc[train_mask, features_primary].values
X_test_b  = df_bayes.loc[test_mask,  features_primary].values
y_train_b = df_bayes.loc[train_mask, 'Outbreak'].values
y_test_b  = df_bayes.loc[test_mask,  'Outbreak'].values

print(f"Training rows:              {X_train_b.shape[0]}")
print(f"Test rows:                  {X_test_b.shape[0]}")
print(f"Training outbreak rate:     {y_train_b.mean():.1%}")
print(f"Test outbreak rate:         {y_test_b.mean():.1%}")
print(f"AP baseline for 2017:       {y_test_b.mean():.3f}")

### STANDARDISE

In [ ]:
# Fit scaler on training set only
scaler_bayes = StandardScaler()
X_train_b_sc = scaler_bayes.fit_transform(X_train_b)
X_test_b_sc  = scaler_bayes.transform(X_test_b)

### BUILD AND SAMPLE THE MODEL

In [ ]:
with pm.Model() as bayes_model:

    # Using weakly informative priors, which is Normal(0, 10), on all coefficients
    intercept = pm.Normal('Intercept', mu=0, sigma=10)
    b_cases = pm.Normal('beta_cases', mu=0, sigma=10)
    b_temp = pm.Normal('beta_temp', mu=0, sigma=10)
    b_precip = pm.Normal('beta_precip', mu=0, sigma=10)

    # Linear predictor
    mu = (intercept + b_cases * X_train_b_sc[:, 0] + b_temp * X_train_b_sc[:, 1] + b_precip * X_train_b_sc[:, 2])

    # Likelihood (binary outbreak outcome)
    p = pm.math.sigmoid(mu)
    likelihood = pm.Bernoulli('likelihood', p=p, observed=y_train_b)

    # Sample using NUTS
    trace = pm.sample(1000, tune=1000, chains=4, target_accept=0.9, random_seed=42, progressbar=True)

### CONVERGENCE CHECK

In [ ]:
print("\n=== GELMAN-RUBIN R-HAT VALUES ===")
rhat = az.rhat(trace)
for var in ['Intercept', 'beta_cases', 'beta_temp', 'beta_precip']:
    print(f"  {var}: {float(rhat[var].values):.4f}")

### POSTERIOR SUMMARY

In [ ]:
print("\n=== POSTERIOR SUMMARY ===")
summary = az.summary(trace, var_names=['Intercept', 'beta_cases', 'beta_temp', 'beta_precip'], hdi_prob=0.95)
print(summary[['mean', 'hdi_2.5%', 'hdi_97.5%']])

### PREDICTION ON TEST SET (2017)

In [ ]:
# Extract posterior means as point estimates
post = trace.posterior
b0  = float(post['Intercept'].values.mean())
b1  = float(post['beta_cases'].values.mean())
b2  = float(post['beta_temp'].values.mean())
b3  = float(post['beta_precip'].values.mean())

# Compute predicted probabilities on 2017 test set
logits   = (b0 + b1 * X_test_b_sc[:, 0] + b2 * X_test_b_sc[:, 1] + b3 * X_test_b_sc[:, 2])
y_prob_b = 1 / (1 + np.exp(-logits))
y_pred_b = (y_prob_b >= 0.5).astype(int)

##### EVALUATE ON TEST SET

In [ ]:
print("\n=== BAYESIAN MODEL PERFORMANCE (Test: 2017) ===")
print(f"AP baseline (outbreak):   {y_test_b.mean():.3f}")
print(f"F2:                         "
      f"{fbeta_score(y_test_b, y_pred_b, beta=2, zero_division=0):.3f}")
print(f"AP:                         "
      f"{average_precision_score(y_test_b, y_prob_b):.3f}")
print(f"F1:                         "
      f"{f1_score(y_test_b, y_pred_b, zero_division=0):.3f}")
print(f"Accuracy:                   "
      f"{accuracy_score(y_test_b, y_pred_b):.3f}")

# BAYESIAN LOGISTIC REGRESSION II (Train 2012-2015, Test 2016)

In [ ]:
# Recomputing outbreak labels using 2012-2016 training years only
df_bayes = compute_outbreak_labels(df_model, train_years=[2012, 2013, 2014, 2015])

### TRAIN/TEST SPLIT

In [ ]:
# Training on 2012-2015, test on 2016
train_mask = df_bayes['Year'].isin([2012, 2013, 2014, 2015])
test_mask  = df_bayes['Year'] == 2016

X_train_b = df_bayes.loc[train_mask, features_primary].values
X_test_b  = df_bayes.loc[test_mask,  features_primary].values
y_train_b = df_bayes.loc[train_mask, 'Outbreak'].values
y_test_b  = df_bayes.loc[test_mask,  'Outbreak'].values

print(f"Training rows:              {X_train_b.shape[0]}")
print(f"Test rows:                  {X_test_b.shape[0]}")
print(f"Training outbreak rate:     {y_train_b.mean():.1%}")
print(f"Test outbreak rate:         {y_test_b.mean():.1%}")
print(f"AP baseline for 2016:       {y_test_b.mean():.3f}")

### STANDARDISE

In [ ]:
# Fit scaler on training set only
scaler_bayes = StandardScaler()
X_train_b_sc = scaler_bayes.fit_transform(X_train_b)
X_test_b_sc  = scaler_bayes.transform(X_test_b)

### BUILD AND SAMPLE THE MODEL

In [ ]:
with pm.Model() as bayes_model:

    # Using weakly informative priors, which is Normal(0, 10), on all coefficients
    intercept = pm.Normal('Intercept', mu=0, sigma=10)
    b_cases = pm.Normal('beta_cases', mu=0, sigma=10)
    b_temp = pm.Normal('beta_temp', mu=0, sigma=10)
    b_precip = pm.Normal('beta_precip', mu=0, sigma=10)

    # Linear predictor
    mu = (intercept + b_cases * X_train_b_sc[:, 0] + b_temp * X_train_b_sc[:, 1] + b_precip * X_train_b_sc[:, 2])

    # Likelihood (binary outbreak outcome)
    p = pm.math.sigmoid(mu)
    likelihood = pm.Bernoulli('likelihood', p=p, observed=y_train_b)

    # Sample using NUTS
    trace = pm.sample(1000, tune=1000, chains=4, target_accept=0.9, random_seed=42, progressbar=True)

### CONVERGENCE CHECK

In [ ]:
print("\n=== GELMAN-RUBIN R-HAT VALUES ===")
rhat = az.rhat(trace)
for var in ['Intercept', 'beta_cases', 'beta_temp', 'beta_precip']:
    print(f"  {var}: {float(rhat[var].values):.4f}")

### POSTERIOR SUMMARY

In [ ]:
print("\n=== POSTERIOR SUMMARY ===")
summary = az.summary(trace, var_names=['Intercept', 'beta_cases', 'beta_temp', 'beta_precip'], hdi_prob=0.95)
print(summary[['mean', 'hdi_2.5%', 'hdi_97.5%']])

### PREDICTION ON TEST SET (2016)

In [ ]:
# Extract posterior means as point estimates
post = trace.posterior
b0  = float(post['Intercept'].values.mean())
b1  = float(post['beta_cases'].values.mean())
b2  = float(post['beta_temp'].values.mean())
b3  = float(post['beta_precip'].values.mean())

# Compute predicted probabilities on 2017 test set
logits   = (b0 + b1 * X_test_b_sc[:, 0] + b2 * X_test_b_sc[:, 1] + b3 * X_test_b_sc[:, 2])
y_prob_b = 1 / (1 + np.exp(-logits))
y_pred_b = (y_prob_b >= 0.5).astype(int)

##### EVALUATE ON TEST SET

In [ ]:
print("\n=== BAYESIAN MODEL PERFORMANCE (Test: 2016) ===")
print(f"AP baseline (outbreak):   {y_test_b.mean():.3f}")
print(f"F2:                         "
      f"{fbeta_score(y_test_b, y_pred_b, beta=2, zero_division=0):.3f}")
print(f"AP:                         "
      f"{average_precision_score(y_test_b, y_prob_b):.3f}")
print(f"F1:                         "
      f"{f1_score(y_test_b, y_pred_b, zero_division=0):.3f}")
print(f"Accuracy:                   "
      f"{accuracy_score(y_test_b, y_pred_b):.3f}")

<h1 style="text-align: center; font-size: 40px;">Nigeria and Benin Combined Map</h1>

In [ ]:
# Natural Earth country boundaries
shpfilename = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
reader = shapereader.Reader(shpfilename)

# Countries to highlight
highlight_colors = {"Nigeria": "green", "Benin": "gold"}

lon_formatter = LongitudeFormatter()
lat_formatter = LatitudeFormatter()

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

# Base map
ax.add_feature(cf.LAND, color="whitesmoke", zorder=0)
ax.add_feature(cf.OCEAN, alpha=0.4, color="lightblue", zorder=1)
ax.add_feature(cf.RIVERS, alpha=0.4, zorder=2)
ax.add_feature(cf.COASTLINE.with_scale("10m"), linewidth=0.8, edgecolor="black", zorder=4)
ax.add_feature(cf.BORDERS.with_scale("10m"), linewidth=0.8, edgecolor="gray", zorder=4)

# Highlight Nigeria and Benin
for record in reader.records():
    name = record.attributes["NAME_EN"]
    if name in highlight_colors:
        ax.add_geometries([record.geometry], ccrs.PlateCarree(), facecolor=highlight_colors[name],
                          edgecolor="black", linewidth=1.5, alpha=0.7, zorder=5)

# Extent covering Nigeria and Benin
ax.set_extent([-4.0, 16.0, 3.0, 15.0], crs=ccrs.PlateCarree())

# Ticks and labels
ax.set_xticks([-2, 2, 6, 10, 14], crs=ccrs.PlateCarree())
ax.set_yticks([4, 6, 8, 10, 12, 14], crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(lon_formatter)
ax.yaxis.set_major_formatter(lat_formatter)

ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.tick_params(labelsize=9)

# Legend
legend_patches = [mpatches.Patch(facecolor=color, edgecolor="black", alpha=0.7, label=country)
                  for country, color in highlight_colors.items()]
ax.legend(handles=legend_patches, fontsize=11, loc="upper left", title="Country", title_fontsize=10)

ax.set_title("Study area: Nigeria and Benin", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("combined_countries_map.png", dpi=300, bbox_inches="tight")
plt.show()